In [1]:
!pip install nltk

import nltk
nltk.download('punkt')

from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter
import numpy as np

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [2]:
corpus = """
I love natural language processing
I love machine learning
natural language processing is interesting
machine learning is powerful
I love artificial intelligence
"""

print("Training Corpus:\n", corpus)


Training Corpus:
 
I love natural language processing
I love machine learning
natural language processing is interesting
machine learning is powerful
I love artificial intelligence



In [4]:
nltk.download('punkt_tab')
tokens = word_tokenize(corpus.lower())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
bigram_model = defaultdict(Counter)

for i in range(len(tokens) - 1):
    bigram_model[tokens[i]][tokens[i+1]] += 1

# Trigram Model
trigram_model = defaultdict(Counter)

for i in range(len(tokens) - 2):
    trigram_model[(tokens[i], tokens[i+1])][tokens[i+2]] += 1

In [6]:
def normalize_model(model):
    prob_model = {}
    for context, counter in model.items():
        total = sum(counter.values())
        prob_model[context] = {word: count/total for word, count in counter.items()}
    return prob_model

bigram_prob = normalize_model(bigram_model)
trigram_prob = normalize_model(trigram_model)

In [11]:
def predict_next_word(input_text, top_k=3):
    words = word_tokenize(input_text.lower())

    # Try trigram first (better context)
    if len(words) >= 2:
        context = (words[-2], words[-1])
        if context in trigram_prob:
            predictions = trigram_prob[context]
            sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
            return sorted_preds[:top_k]

    # Fallback to bigram
    if len(words) >= 1:
        context = words[-1]
        if context in bigram_prob:
            predictions = bigram_prob[context]
            sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
            return sorted_preds[:top_k]

    return [("No prediction", 0)]

In [12]:
test_sentences = [
    "i love",
    "machine learning",
    "natural language",
    "i love machine"
]

print("\n--- Auto-complete Predictions ---")

for sentence in test_sentences:
    print(f"\nInput: '{sentence}'")
    preds = predict_next_word(sentence)
    print("Predictions:", preds)


--- Auto-complete Predictions ---

Input: 'i love'
Predictions: [('natural', 0.3333333333333333), ('machine', 0.3333333333333333), ('artificial', 0.3333333333333333)]

Input: 'machine learning'
Predictions: [('natural', 0.5), ('is', 0.5)]

Input: 'natural language'
Predictions: [('processing', 1.0)]

Input: 'i love machine'
Predictions: [('learning', 1.0)]


In [13]:
user_input = input("\nEnter a sentence: ")
predictions = predict_next_word(user_input)

print("\nTop Predictions:")
for word, prob in predictions:
    print(f"{word} (probability: {round(prob, 3)})")


Enter a sentence: HOW YOU DOING

Top Predictions:
No prediction (probability: 0)
